In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q17/annotated/checkpoints/pre_cell_2.pickle

trying: ['part']
me:  2
trying: ['lineitem']


me:  1
trying: ['pd']
me:  0
trying: ['load_part']
me:  2
trying: ['STORAGE_OPTS']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['load_lineitem']
me:  1
trying: ['tpch_parent']
me:  0


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable part
Declaring variable load_part


In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Cell 2 optimized for cudf

# 1. Filter PART to get the relevant keys on GPU using boolean indexing instead of .query
filtered_keys = part[(part.P_BRAND == 'Brand#23') & (part.P_CONTAINER == 'MED BOX')].P_PARTKEY

# 2. Early‐filter LINEITEM to just those keys, then compute the 20%‐scaled average quantity per key on GPU
li_subset = lineitem[lineitem.L_PARTKEY.isin(filtered_keys)]
avg_map = li_subset.groupby('L_PARTKEY')['L_QUANTITY'].mean() * 0.2

# 3. Merge the precomputed avg back into the subset (avoiding .map) and select only the needed cols
li_with_avg = li_subset[['L_PARTKEY', 'L_QUANTITY', 'L_EXTENDEDPRICE']]  
li_with_avg = li_with_avg.merge(avg_map.rename('avg'), 
                                 left_on='L_PARTKEY', 
                                 right_index=True)

# 4. Apply the quantity filter and compute the final metric on GPU
filtered = li_with_avg[li_with_avg.L_QUANTITY < li_with_avg.avg]
total = pd.DataFrame({
    'avg_yearly': [filtered.L_EXTENDEDPRICE.sum() / 7.0]
})

CPU times: user 83.8 ms, sys: 25.5 ms, total: 109 ms
Wall time: 115 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q17/rewritten/o4_mini_high/checkpoints/post_cell_2_try_2.pickle

migration speed (bps): 799301986.5364224
---------------------------
variables to migrate:
filtered_keys 48
DATA_ROOT 80
filtered 48
lineitem 48
STORAGE_OPTS 64
load_lineitem 144
total 48
part 48
load_part 144
li_subset 48
pd 72
li_with_avg 48
avg_map 48
tpch_parent 54
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q17/rewritten/o4_mini_high/checkpoints/post_cell_2_try_2.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['filtered_keys', 'filtered', 'li_subset', 'avg_map', 'total', 'li_with_avg']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q17/opt_cell_exec_info_2_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q17/annotated/checkpoints/post_cell_2.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
